# 00. Rustamps beginner walkthrough

Rustamps runs the StaMPS persistent-scatterer workflow as one Rust command. This notebook runs that command on a
real patch and checks, with independent tools, that it did what it claims. Every cell that makes a claim ends in an
assertion, so a notebook that runs to the end has proven those claims.

Words you will see: a **dataset** is a folder of radar-derived files; a **patch** is one tile of it; a **stage** is one
of eight processing steps; a **persistent scatterer (PS)** is a ground point stable enough to measure.

> **Scope.** These notebooks prove Stages 1-5 on `PATCH_1` of the golden dataset, starting from the raw
> `pscands.1.*` candidate files and ending at the Stage-5 patch products. Stages 6-8 are **not covered**:
> they need the merged four-patch golden data, which is not part of the local download.

In [ ]:
import sys
from pathlib import Path

here = Path.cwd().resolve()
root = next(p for p in (here, *here.parents) if (p / "Cargo.toml").exists())
sys.path[:0] = [str(root), str(root / "notebooks")]

import matplotlib.pyplot as plt, numpy as np, pandas as pd
from IPython.display import display, Markdown
import rustamps_nb as nb
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 110); pd.set_option("display.max_rows", 120)
print("binary:", nb.binary()); print("golden:", nb.require_golden())

## 1. The program starts and reports a version

In [ ]:
rc, out, err, _ = nb.cli("--version")
assert out.startswith("rustamps "), out
proof = [nb.check(True, "the rustamps binary runs", out.strip())]

## 2. `status` reads a dataset without changing it

`status` is read-only. We check that by listing the folder before and after.

In [ ]:
def listing(root):
    return sorted((str(p.relative_to(root)), p.stat().st_size, p.stat().st_mtime_ns) for p in root.rglob("*") if p.is_file())
before = listing(nb.GOLDEN / "PATCH_1")
status, _ = nb.cli_json("status", "--dataset", nb.GOLDEN)
after = listing(nb.GOLDEN / "PATCH_1")
display(pd.DataFrame(status["patches"]))
p1 = next(p for p in status["patches"] if p["patch"] == "PATCH_1")
proof.append(nb.check(before == after, "status did not touch the dataset", f"{len(before)} files unchanged"))
proof.append(nb.check(p1["stage"] == 5, "golden PATCH_1 is recognised as completed through stage 5", str(p1)))

## 3. `run --dry-run` shows the plan and writes nothing

We stage a scratch copy of the PATCH_1 inputs (the golden folder is never written to).

In [ ]:
run = nb.stage_inputs("beginner")
before = listing(run)
plan, _ = nb.cli_json("run", "--dataset", run, "--start-step", 1, "--end-step", 8, "--dry-run")
display(pd.DataFrame(plan)[["stage", "scope", "target", "status", "details"]])
proof.append(nb.check(listing(run) == before, "dry-run wrote nothing", f"{len(before)} files unchanged"))
proof.append(nb.check(sorted({p["stage"] for p in plan}) == list(range(1, 9)) and all(p["status"] == "planned" for p in plan),
                      "dry-run plans stages 1..8", ", ".join(p["details"] for p in plan[:3]) + " ..."))

## 4. Run Stage 1 for real and check it against facts we can count ourselves

Stage 1 turns the raw candidate list into `ps1.mat`. The number of candidates must equal the number of lines in
`pscands.1.ij`, which we count here without Rust.

In [ ]:
summary, seconds = nb.run_stage(run, 1)
n_lines = sum(1 for _ in open(run / "PATCH_1" / "pscands.1.ij"))
ps1 = nb.read_mat(run / "PATCH_1" / "ps1.mat")
golden_ps1 = nb.read_mat(nb.GOLDEN / "PATCH_1" / "ps1.mat")
n_ps = int(ps1["n_ps"].ravel()[0])
proof.append(nb.check(n_ps == n_lines == 81428, "ps1.mat holds every candidate in pscands.1.ij", f"{n_ps} == {n_lines}"))
proof.append(nb.check(np.array_equal(ps1["ij"], golden_ps1["ij"]) and np.array_equal(ps1["xy"], golden_ps1["xy"]),
                      "candidate indices and local coordinates equal the original StaMPS output", f"{seconds:.2f} s"))

## 5. Verdict

In [ ]:
verdict = pd.DataFrame(proof); display(verdict); assert verdict.ok.all(); print("NOTEBOOK PASSED:", len(verdict), "checks")